In [ ]:
import os
import pandas as pd
import numpy as np
import joblib

#### Constants

In [ ]:
str_bucket = 'assessment-alt'
str_task = '04_preprocessing'
str_target = 'price'
str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)
print(f'Bucket: {str_bucket}')
print(f'Task: {str_task}')

#### Load Data

In [ ]:
%%time
df_train = pd.read_csv(f's3://{str_bucket}/03_data_split/train.csv')
df_valid = pd.read_csv(f's3://{str_bucket}/03_data_split/valid.csv')
df_test = pd.read_csv(f's3://{str_bucket}/03_data_split/test.csv')
print(f'Train: {df_train.shape}')
print(f'Valid: {df_valid.shape}')
print(f'Test:  {df_test.shape}')

#### TextNormalizer

In [ ]:
class TextNormalizer:
    def __init__(self):
        pass

    def fit(self, X):
        print('TextNormalizer fit...')
        return self

    def transform(self, X):
        print('TextNormalizer transform...')
        X = X.copy()
        X['subject'] = X['subject'].str.lower().str.strip()
        X['brand'] = X['brand'].str.lower().str.strip()
        X['variety'] = X['variety'].str.lower().str.strip().fillna('base')
        X['card_number'] = X['card_number'].astype(str).str.lower().str.strip()
        X['grading_company'] = X['grading_company'].str.lower().str.strip()
        return X

#### CardKeyFeatures

Creates a card key from the fungibility tuple and computes card-level statistics from training data. This is a **stateful** transformer — `fit()` learns price statistics from training data.

In [ ]:
class CardKeyFeatures:
    def __init__(self):
        self.df_card_stats = None

    def fit(self, X):
        print('CardKeyFeatures fit...')
        X = X.copy()
        X['str_card_key'] = (
            X['year'].astype(str) + '|' +
            X['subject'] + '|' +
            X['brand'] + '|' +
            X['variety'] + '|' +
            X['card_number'] + '|' +
            X['grade'].astype(str) + '|' +
            X['grading_company']
        )
        self.df_card_stats = X.groupby('str_card_key')['price'].agg(
            int_card_txn_count='count',
            flt_card_median_price='median',
            flt_card_mean_price='mean'
        ).reset_index()
        print(f'  Learned stats for {len(self.df_card_stats):,} unique cards')
        return self

    def transform(self, X):
        print('CardKeyFeatures transform...')
        X = X.copy()
        X['str_card_key'] = (
            X['year'].astype(str) + '|' +
            X['subject'] + '|' +
            X['brand'] + '|' +
            X['variety'] + '|' +
            X['card_number'] + '|' +
            X['grade'].astype(str) + '|' +
            X['grading_company']
        )
        X = X.merge(self.df_card_stats, on='str_card_key', how='left')
        X['int_card_txn_count'] = X['int_card_txn_count'].fillna(0).astype(int)
        X['flt_card_median_price'] = X['flt_card_median_price'].fillna(0)
        X['flt_card_mean_price'] = X['flt_card_mean_price'].fillna(0)
        return X

#### SubjectFeatures

Computes subject-level price statistics from training data. **Stateful**.

In [ ]:
class SubjectFeatures:
    def __init__(self):
        self.df_subject_stats = None

    def fit(self, X):
        print('SubjectFeatures fit...')
        self.df_subject_stats = X.groupby('subject')['price'].agg(
            int_subject_txn_count='count',
            flt_subject_median_price='median'
        ).reset_index()
        print(f'  Learned stats for {len(self.df_subject_stats):,} unique subjects')
        return self

    def transform(self, X):
        print('SubjectFeatures transform...')
        X = X.copy()
        X = X.merge(self.df_subject_stats, on='subject', how='left')
        X['int_subject_txn_count'] = X['int_subject_txn_count'].fillna(0).astype(int)
        X['flt_subject_median_price'] = X['flt_subject_median_price'].fillna(0)
        return X

#### BrandFeatures

Computes brand-level price statistics from training data. **Stateful**.

In [ ]:
class BrandFeatures:
    def __init__(self):
        self.df_brand_stats = None

    def fit(self, X):
        print('BrandFeatures fit...')
        self.df_brand_stats = X.groupby('brand')['price'].agg(
            int_brand_txn_count='count',
            flt_brand_median_price='median'
        ).reset_index()
        print(f'  Learned stats for {len(self.df_brand_stats):,} unique brands')
        return self

    def transform(self, X):
        print('BrandFeatures transform...')
        X = X.copy()
        X = X.merge(self.df_brand_stats, on='brand', how='left')
        X['int_brand_txn_count'] = X['int_brand_txn_count'].fillna(0).astype(int)
        X['flt_brand_median_price'] = X['flt_brand_median_price'].fillna(0)
        return X

#### GradeFeatures

Creates grade-related features. **Stateless**.

In [ ]:
class GradeFeatures:
    def __init__(self):
        pass

    def fit(self, X):
        print('GradeFeatures fit...')
        return self

    def transform(self, X):
        print('GradeFeatures transform...')
        X = X.copy()
        X['flt_grade'] = X['grade'].astype(float)
        X['int_is_psa'] = (X['grading_company'] == 'psa').astype(int)
        X['int_is_gem_mint'] = (X['grade'] == 10.0).astype(int)
        return X

#### TimeFeatures

Creates time-based features from the transaction date. **Stateless**.

In [ ]:
class TimeFeatures:
    def __init__(self):
        self.dt_first_txn = pd.Timestamp('2018-05-16')

    def fit(self, X):
        print('TimeFeatures fit...')
        return self

    def transform(self, X):
        print('TimeFeatures transform...')
        X = X.copy()
        X['date'] = pd.to_datetime(X['date'])
        X['int_year_sold'] = X['date'].dt.year
        X['int_month_sold'] = X['date'].dt.month
        X['int_days_since_first_txn'] = (X['date'] - self.dt_first_txn).dt.days
        return X

#### TargetTransform

Creates log-transformed price target. **Stateless**.

In [ ]:
class TargetTransform:
    def __init__(self):
        pass

    def fit(self, X):
        print('TargetTransform fit...')
        return self

    def transform(self, X):
        print('TargetTransform transform...')
        X = X.copy()
        X['flt_log_price'] = np.log1p(X['price'])
        return X

#### PrepareFeatures

Selects the final feature set for modeling. **Stateful** — stores the feature column list.

In [ ]:
class PrepareFeatures:
    def __init__(self):
        self.list_str_features = [
            'flt_grade', 'int_is_psa', 'int_is_gem_mint',
            'int_card_txn_count', 'flt_card_median_price', 'flt_card_mean_price',
            'int_subject_txn_count', 'flt_subject_median_price',
            'int_brand_txn_count', 'flt_brand_median_price',
            'int_year_sold', 'int_month_sold', 'int_days_since_first_txn',
        ]

    def fit(self, X):
        print('PrepareFeatures fit...')
        print(f'  Feature columns: {len(self.list_str_features)}')
        return self

    def transform(self, X):
        print('PrepareFeatures transform...')
        X = X.copy()
        list_str_keep = self.list_str_features + ['flt_log_price']
        X = X[list_str_keep]
        return X

#### Build Pipeline

In [ ]:
list_cls_transformers = [
    TextNormalizer(),
    CardKeyFeatures(),
    SubjectFeatures(),
    BrandFeatures(),
    GradeFeatures(),
    TimeFeatures(),
    TargetTransform(),
    PrepareFeatures(),
]
print(f'Pipeline has {len(list_cls_transformers)} transformers')

#### Fit Pipeline

In [ ]:
%%time
df_tmp = df_train.copy()
for cls_transformer in list_cls_transformers:
    cls_transformer.fit(df_tmp)
    df_tmp = cls_transformer.transform(df_tmp)
print(f'\nTrain shape after pipeline: {df_tmp.shape}')
df_tmp.head()

#### Transform All Splits

In [ ]:
%%time
class PreprocessingModel:
    def __init__(self, list_cls_transformers):
        self.list_cls_transformers = list_cls_transformers

    def transform(self, X):
        for cls_transformer in self.list_cls_transformers:
            X = cls_transformer.transform(X)
        return X

cls_preprocessing = PreprocessingModel(list_cls_transformers)

df_train_processed = cls_preprocessing.transform(df_train)
print(f'Train: {df_train_processed.shape}')

df_valid_processed = cls_preprocessing.transform(df_valid)
print(f'Valid: {df_valid_processed.shape}')

df_test_processed = cls_preprocessing.transform(df_test)
print(f'Test:  {df_test_processed.shape}')

In [ ]:
# Separate features and target
str_target_col = 'flt_log_price'
list_str_features = [c for c in df_train_processed.columns if c != str_target_col]

X_train = df_train_processed[list_str_features]
y_train = df_train_processed[[str_target_col]]

X_valid = df_valid_processed[list_str_features]
y_valid = df_valid_processed[[str_target_col]]

X_test = df_test_processed[list_str_features]
y_test = df_test_processed[[str_target_col]]

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')
print(f'\nFeatures: {list_str_features}')

#### Save

In [ ]:
%%time
# Save preprocessing model locally
joblib.dump(cls_preprocessing, f'{str_dirname_output}/preprocessing_model.joblib')
print(f'Saved preprocessing_model.joblib to {str_dirname_output}')

# Save processed data to S3
for str_name, df_x, df_y in [('train', X_train, y_train),
                               ('valid', X_valid, y_valid),
                               ('test', X_test, y_test)]:
    str_s3_path_x = f's3://{str_bucket}/{str_task}/X_{str_name}.csv'
    str_s3_path_y = f's3://{str_bucket}/{str_task}/y_{str_name}.csv'
    df_x.to_csv(str_s3_path_x, index=False)
    df_y.to_csv(str_s3_path_y, index=False)
    print(f'Saved X_{str_name}.csv ({df_x.shape}) and y_{str_name}.csv ({df_y.shape}) to S3')

#### Takeaways

- **8 transformers** in the pipeline: text normalization, card/subject/brand stats, grade features, time features, target transform, and feature selection
- **Stateful transformers** (CardKeyFeatures, SubjectFeatures, BrandFeatures) learn statistics from training data only — preventing data leakage
- **13 numeric features** produced for modeling
- **Log-transformed price** (flt_log_price) used as target to handle the heavy right skew
- Cards not seen in training get 0 for card-level stats — the model must learn to handle these cold start cases
- Preprocessing model saved as joblib for reproducible inference